In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
from datetime import timedelta

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], "GPU")
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print("GPU setup warning:", e)

# ============================================================
# 1. Wczytanie danych
# ============================================================

csv_path = r"C:\Users\tomas\Desktop\github\upp_csde\metody_si\projekt\dane\merged_data.csv"

df_raw = pd.read_csv(csv_path, encoding="utf-8")

if df_raw.empty:
    raise ValueError(f"DataFrame loaded from {csv_path} is empty.")

if "POST" not in df_raw.columns:
    raise ValueError("Brak kolumny POST w danych.")

# 2. Data
df_raw["DATA"] = pd.to_datetime(dict(year=df_raw["ROK"], month=df_raw["MC"], day=df_raw["DZ"]))

# 3. Sortowanie
df_raw = df_raw.sort_values(["POST", "DATA"]).reset_index(drop=True)

# 4. SEZONOWOSC: cechy czasu
df_raw["month"] = df_raw["DATA"].dt.month
df_raw["dayofyear"] = df_raw["DATA"].dt.dayofyear

# 5. Target = srednia temperatura z nastepnego dnia
df_raw["TARGET"] = df_raw.groupby("POST")["STD"].shift(-1)

# 6. Lagi
df_raw["STD_lag1"] = df_raw.groupby("POST")["STD"].shift(1)
df_raw["TMAX_lag1"] = df_raw.groupby("POST")["TMAX"].shift(1)
df_raw["TMIN_lag1"] = df_raw.groupby("POST")["TMIN"].shift(1)

# 7. Usuniecie brakow
need_cols = ["STD_lag1", "TMAX_lag1", "TMIN_lag1", "TARGET", "POST", "month", "dayofyear"]
df = df_raw.dropna(subset=need_cols).copy()

if df.empty:
    raise ValueError("No data left after creating lags / dropping NA.")

# 8. Kodowanie lokalizacji
le = LabelEncoder()
df["POST_ID"] = le.fit_transform(df["POST"])

num_locations = df["POST_ID"].nunique()
embedding_dim = min(16, max(4, num_locations // 2))

print("Liczba lokalizacji:", num_locations)
print("Embedding dim:", embedding_dim)

# 9. Cechy numeryczne (z sezonowoscia)
num_features = [
    "STD_lag1",
    "TMAX_lag1",
    "TMIN_lag1",
    "month",
    "dayofyear"
]

X_num = df[num_features].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)
X_loc = df["POST_ID"].values.astype(np.int32)
y = pd.to_numeric(df["TARGET"], errors="coerce").values.astype(np.float32)

if np.isnan(X_num).any() or np.isnan(y).any():
    raise ValueError("W danych sa NaN po konwersji do liczb.")

# 10. Podzial czasowy
split = int(len(df) * 0.8)
if split < 1:
    split = 1 if len(df) > 1 else len(df)

X_num_train, X_num_test = X_num[:split], X_num[split:]
X_loc_train, X_loc_test = X_loc[:split], X_loc[split:]
y_train, y_test = y[:split], y[split:]

if X_num_train.shape[0] == 0:
    raise ValueError("Training set is empty after split.")

if X_num_test.shape[0] == 0:
    if X_num_train.shape[0] > 1:
        X_num_test = X_num_train[-1:].copy()
        X_loc_test = X_loc_train[-1:].copy()
        y_test = y_train[-1:].copy()

        X_num_train = X_num_train[:-1]
        X_loc_train = X_loc_train[:-1]
        y_train = y_train[:-1]
    else:
        raise ValueError("Not enough data for testing.")

# 11. Skalowanie
mean = X_num_train.mean(axis=0)
std_dev = X_num_train.std(axis=0)
std_fixed = np.where(std_dev == 0, 1.0, std_dev)

X_num_train = ((X_num_train - mean) / std_fixed).astype(np.float32)
X_num_test = ((X_num_test - mean) / std_fixed).astype(np.float32)

# 12. Model
num_input = keras.Input(shape=(X_num_train.shape[1],), name="num_input")
loc_input = keras.Input(shape=(1,), dtype="int32", name="loc_input")

loc_embedding = layers.Embedding(
    input_dim=num_locations,
    output_dim=embedding_dim,
    name="location_embedding"
)(loc_input)

loc_vec = layers.Flatten()(loc_embedding)

x = layers.Concatenate()([num_input, loc_vec])
x = layers.Dense(32, activation="relu")(x)
x = layers.Dense(16, activation="relu")(x)
x = layers.Dense(8, activation="relu")(x)
output = layers.Dense(1)(x)

model = keras.Model(inputs=[num_input, loc_input], outputs=output)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# 13. Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

# 14. Trenowanie
fit_kwargs = dict(
    x={"num_input": X_num_train, "loc_input": X_loc_train},
    y=y_train,
    epochs=200,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

if X_num_train.shape[0] >= 10:
    fit_kwargs["validation_split"] = 0.2
else:
    fit_kwargs["validation_data"] = (
        {"num_input": X_num_test, "loc_input": X_loc_test},
        y_test
    )

history = model.fit(**fit_kwargs)

# 15. Predykcja testowa
y_pred = model.predict(
    {"num_input": X_num_test, "loc_input": X_loc_test},
    verbose=0
).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("\nMAE test:", round(mae, 3))
print("RMSE test:", round(rmse, 3))

# 16. Zapis modelu
model.save("temperature_model_embedding_security.keras")


# ============================================================
# FUNKCJE DO PROGNOZY
# ============================================================

def predict_temperature_numeric(features, post_id):
    """
    Predykcja dla podanych cech numerycznych i ID lokalizacji.
    features: array [STD_lag1, TMAX_lag1, TMIN_lag1, month, dayofyear]
    """
    if post_id not in le.classes_:
        raise ValueError(f"Lokalizacja nie istnieje: {post_id}")

    post_id_num = le.transform([post_id])[0]

    X_num_incoming = np.array([features], dtype=np.float32)
    X_loc_manual = np.array([[post_id_num]], dtype=np.int32)

    X_num_scaled = ((X_num_incoming - mean) / std_fixed).astype(np.float32)

    prediction = model.predict(
        {"num_input": X_num_scaled, "loc_input": X_loc_manual},
        verbose=0
    ).flatten()[0]

    return float(prediction)


def get_last_known_row(post_name):
    """Bierze ostatni rekord z df_raw dla danej lokalizacji."""
    loc_df = df_raw[df_raw["POST"] == post_name].sort_values("DATA")

    if loc_df.empty:
        raise ValueError(f"Brak danych dla lokalizacji: {post_name}")

    last_row = loc_df.iloc[-1]

    return {
        "date": pd.to_datetime(last_row["DATA"]),
        "std": float(last_row["STD"]),
        "tmax": float(last_row["TMAX"]),
        "tmin": float(last_row["TMIN"]),
        "month": int(last_row["month"]),
        "dayofyear": int(last_row["dayofyear"])
    }


def forecast_next_days(post_name, days=7):
    """
    Prognoza na danej liczbe dni od ostatniej dostepnej daty w CSV.
    """
    last_row = get_last_known_row(post_name)

    last_known_date = pd.to_datetime(last_row["date"])
    current_date = last_known_date + timedelta(days=1)

    std_lag1 = last_row["std"]
    tmax_lag1 = last_row["tmax"]
    tmin_lag1 = last_row["tmin"]

    results = []

    print("\n" + "=" * 70)
    print(f"PROGNOZA NA {days} DNI DLA: {post_name}")
    print("=" * 70)
    print(f"Ostatnia data w CSV: {last_known_date.date()}")
    print(f"Start prognozy: {current_date.date()}")
    print(f"Liczba dni prognozy: {days}")
    print("-" * 70)

    for step in range(days):
        month_now = current_date.month
        dayofyear_now = current_date.timetuple().tm_yday

        features = [
            std_lag1,
            tmax_lag1,
            tmin_lag1,
            month_now,
            dayofyear_now
        ]

        pred_std = predict_temperature_numeric(features, post_name)

        results.append({
            "data": current_date,
            "POST": post_name,
            "STD_lag1_used": std_lag1,
            "TMAX_lag1_used": tmax_lag1,
            "TMIN_lag1_used": tmin_lag1,
            "month": month_now,
            "dayofyear": dayofyear_now,
            "predicted_STD": pred_std
        })

        print(
            f"{current_date.date()} | "
            f"STYCZEN={month_now:2d} DOY={dayofyear_now:3d} | "
            f"wejscie: STD={std_lag1:6.2f}, TMAX={tmax_lag1:6.2f}, TMIN={tmin_lag1:6.2f} "
            f"-> prognoza STD={pred_std:6.2f}"
        )

        std_lag1 = pred_std
        tmax_lag1 = pred_std + 2.0
        tmin_lag1 = pred_std - 2.0

        current_date += timedelta(days=1)

    forecast_df = pd.DataFrame(results)

    print("-" * 70)
    print("Wynik prognozy:")
    print(forecast_df[["data", "POST", "predicted_STD"]].to_string(index=False))

    return forecast_df


TensorFlow: 2.10.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Liczba lokalizacji: 179
Embedding dim: 16
Epoch 1/200
2656/2656 [==============================] - 9s 3ms/step - loss: 13.7854 - mae: 2.6954 - val_loss: 56.1589 - val_mae: 6.7031
Epoch 2/200
2656/2656 [==============================] - 11s 4ms/step - loss: 9.4462 - mae: 2.3875 - val_loss: 48.1329 - val_mae: 6.1054
Epoch 3/200
2656/2656 [==============================] - 11s 4ms/step - loss: 9.3807 - mae: 2.3777 - val_loss: 39.6143 - val_mae: 5.4586
Epoch 4/200
2656/2656 [==============================] - 11s 4ms/step - loss: 9.3524 - mae: 2.3742 - val_loss: 31.5809 - val_mae: 4.7882
Epoch 5/200
2656/2656 [==============================] - 11s 4ms/step - loss: 9.3306 - mae: 2.3715 - val_loss: 27.1128 - val_mae: 4.3757
Epoch 6/200
2656/2656 [==============================] - 11s 4ms/step - loss: 9.3061 - mae: 2.3685 - val_loss: 22.9459 - val_mae: 3.9802
Epoch 7/200
2656/2656 [======================

In [7]:
# ============================================================
# PRZYKLAD UZYCIA
# ============================================================
prognoza = forecast_next_days(
    post_name="JAROCIN",
    days=7
)

print("\nPelny DataFrame prognozy:")
print(prognoza)


PROGNOZA NA 7 DNI DLA: JAROCIN
Ostatnia data w CSV: 2025-12-31
Start prognozy: 2026-01-01
Liczba dni prognozy: 7
----------------------------------------------------------------------
2026-01-01 | STYCZEN= 1 DOY=  1 | wejscie: STD= -3.30, TMAX= -1.90, TMIN= -3.90 -> prognoza STD= -5.18
2026-01-02 | STYCZEN= 1 DOY=  2 | wejscie: STD= -5.18, TMAX= -3.18, TMIN= -7.18 -> prognoza STD= -5.50
2026-01-03 | STYCZEN= 1 DOY=  3 | wejscie: STD= -5.50, TMAX= -3.50, TMIN= -7.50 -> prognoza STD= -5.77
2026-01-04 | STYCZEN= 1 DOY=  4 | wejscie: STD= -5.77, TMAX= -3.77, TMIN= -7.77 -> prognoza STD= -5.97
2026-01-05 | STYCZEN= 1 DOY=  5 | wejscie: STD= -5.97, TMAX= -3.97, TMIN= -7.97 -> prognoza STD= -6.14
2026-01-06 | STYCZEN= 1 DOY=  6 | wejscie: STD= -6.14, TMAX= -4.14, TMIN= -8.14 -> prognoza STD= -6.28
2026-01-07 | STYCZEN= 1 DOY=  7 | wejscie: STD= -6.28, TMAX= -4.28, TMIN= -8.28 -> prognoza STD= -6.39
----------------------------------------------------------------------
Wynik prognozy:
      d